# BigQuery codes to create tables from Openalex snapshot

OpenAlex snapshot: https://openalex.s3.amazonaws.com/browse.html \
Openalex snapshot used in the paper: updated Oct 2025.

## Works

Creation of 3 tables from works table:
* works_info
* works_authorships
* works_topics

### works_info

Table Name: `project_name.dataset_name.works_info` \
Purpose: Contains works information. \
Partitioned by: publication_year \
Clustered by: type \
Rows: Each row correspond to one paper. We are discarding papers with missing publication year, that are not articles or preprints (such as book chapters or repositories), that have not been published in journal or conferences. \ 
Columns:
- `work_id` (INT64): Openalex unique paper ID. Identifies the paper in the Openalex database.
- `is_retracted` (INT64): 1 if the paper has been retracted, 0 otherwise.
- `doi` (STRING): The Digital Object Identifier (DOI) of the paper, a unique alphanumeric string to identify the paper online.
- `publication_year` (DATE): The year in which the paper was published.
- `publication_date` (DATE): The date in which the paper was published, formatted as an ISO 8601 date.
- `title` (STRING): The title of the paper.
- `language` (STRING): The language in which the paper is written.
- `type` (STRING): The type of paper, either 'article' (peer-reviewed) or 'preprint' (pre-publication version).
- `fwci` (FLOAT64): The Field Weighted Citation Impact of paper.
- `cited_by_count` (INT64): The total number of times the paper has been cited by other works (up to the date of the considered snapshot).
- `referenced_works_count` (INT64): The number of references cited by the paper.
- `authors_count` (INT64): The total number of authors who contributed to the paper.
- `journal_id` (INT64): The Openalex unique ID of the journal in which the paper was published.
- `journal_name` (STRING): The name of the journal in which the paper was published.
- `updated_date` (TIMESTAMP): The last time anything in this Work object changed, expressed as an ISO 8601 date string (in UTC). This date is updated for any change at all, including increases in various counts.

In [ ]:
CREATE TABLE `project_name.dataset_name.works_info`
PARTITION BY publication_year
AS

WITH sources_info_filtered AS (
  SELECT source_id
  FROM mobs-jessica.openalex.sources_info 
  WHERE source_type IN ('journal','conference')
),

works_info_retracted_test AS (
  SELECT
  CAST(REGEXP_EXTRACT(json_value(work, '$.id'), r'W(\d+)') AS INT64) AS work_id,
  CAST(json_value(work, '$.is_retracted') = 'true' AS INT64) AS is_retracted,
  CAST(json_value(work, '$.doi') AS STRING) AS doi,
  --- CAST(EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', json_value(work, '$.publication_date'))) AS INT64) AS publication_year,
  CAST(CONCAT(CAST(EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', json_value(work, '$.publication_date'))) AS STRING), '-01-01') AS DATE) AS publication_year,
  CAST(json_value(work, '$.publication_date') AS DATE) AS publication_date,
  CAST(json_value(work, '$.title') AS STRING) AS title,
  CAST(json_value(work, '$.language') AS STRING) AS language,
  CAST(json_value(work, '$.type') AS STRING) AS type,
  CAST(json_value(work, '$.fwci') AS FLOAT64) AS fwci,
  CAST(json_value(work, '$.cited_by_count') AS INT64) AS cited_by_count,
  CAST(json_value(work, '$.referenced_works_count') AS INT64) AS referenced_works_count,
  CAST(IFNULL(ARRAY_LENGTH(JSON_EXTRACT_ARRAY(work, '$.authorships')), 0) AS INT64) AS authors_count,
  CAST(
    IF(
      JSON_EXTRACT(work, '$.primary_location') IS NOT NULL AND 
      JSON_EXTRACT(work, '$.primary_location.source') IS NOT NULL,
      REGEXP_EXTRACT(JSON_EXTRACT_SCALAR(JSON_EXTRACT(work, '$.primary_location.source'), '$.id'), r'S(\d+)'),
      NULL
    ) 
  AS INT64) AS journal_id,
  CAST(
    IF(
      JSON_EXTRACT(work, '$.primary_location') IS NOT NULL AND 
      JSON_EXTRACT(work, '$.primary_location.source') IS NOT NULL,
      JSON_EXTRACT_SCALAR(JSON_EXTRACT(work, '$.primary_location.source'), '$.display_name'),
      NULL
    ) 
  AS STRING) AS journal_name
  FROM 
    `project_name.dataset_name.works`
  WHERE 
    json_value(work, '$.publication_year') IS NOT NULL  
    AND json_value(work, '$.publication_year') != 'NaN'
    AND json_value(work, '$.type') IN ('article', 'preprint')
    AND LENGTH(CAST(EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', json_value(work, '$.publication_date'))) AS STRING)) = 4
)

SELECT
  *
FROM works_info_retracted_test
WHERE journal_id IN (SELECT source_id FROM sources_info_filtered);

### works_authorships

Table Name: `project_name.dataset_name.works_authorships` \
Purpose: Contains works authorships information. \
Partitioned by: publication_year \
Clustered by: work_id, institution_id \
Rows: There is one row for each paper, each author, each institution. We are discarding papers with missing publication year, that are not articles or preprints (such as book chapters or repositories), that have not been published in journal or conferences, and with missing authorship information. Note that an author can have multiple institutions, and that an author can have missing instituion information. \
Columns: 
- `work_id` (INT64): Openalex unique paper ID. Identifies the paper in the Openalex database.
- `is_retracted` (INT64): 1 if the paper has been retracted, 0 otherwise.
- `publication_year` (DATE): The year in which the paper was published.
- `author_id` (INT64): Unique identifier for the author in the Openalex database.
- `author_name` (STRING): Name of the author who contributed to the paper.
- `institution_id` (INT64): Unique identifier for the institution where the author is affiliated.
- `institution_name` (STRING): Name of the institution where the author is affiliated.
- `country_code` (STRING): The ISO country code of the institution's location.


In [ ]:
CREATE OR REPLACE TABLE `project_name.dataset_name.works_authorships`
PARTITION BY publication_year
CLUSTER BY work_id, institution_id
AS

WITH
-- Step 1: Filter sources (journals & conferences)
sources_info_filtered AS (
  SELECT source_id
  FROM `project_name.dataset_name.sources_info`
  WHERE source_type IN ('journal', 'conference')
),

-- Step 2: Parse and filter `works_test` records before UNNEST
filtered_works AS (
  SELECT
    CAST(REGEXP_EXTRACT(JSON_VALUE(work, '$.id'), r'W(\d+)') AS INT64) AS work_id,
    CAST(JSON_VALUE(work, '$.is_retracted') = 'true' AS INT64) AS is_retracted,
    PARSE_DATE('%Y-%m-%d', JSON_VALUE(work, '$.publication_date')) AS publication_date,
    JSON_EXTRACT_ARRAY(work, '$.authorships') AS authorships,
    JSON_EXTRACT(work, '$.primary_location.source') AS primary_source
  FROM `mobs-jessica.openalex.works_test`
  WHERE
    JSON_VALUE(work, '$.type') IN ('article', 'preprint')
    AND JSON_VALUE(work, '$.publication_year') IS NOT NULL
    AND JSON_VALUE(work, '$.publication_year') != 'NaN'
    AND SAFE.PARSE_DATE('%Y-%m-%d', JSON_VALUE(work, '$.publication_date')) IS NOT NULL
),

-- Step 3: Unnest authorships
authorships_exploded AS (
  SELECT
    fw.work_id,
    fw.is_retracted,
    DATE_TRUNC(fw.publication_date, YEAR) AS publication_year,
    CAST(REGEXP_EXTRACT(JSON_VALUE(a, '$.author.id'), r'A(\d+)') AS INT64) AS author_id,
    CAST(JSON_VALUE(a, '$.author.display_name') AS STRING) AS author_name,
    JSON_EXTRACT_ARRAY(a, '$.institutions') AS institutions,
    fw.primary_source
  FROM filtered_works fw,
  UNNEST(fw.authorships) AS a
),

-- Step 4: Unnest institutions
institutions_exploded AS (
  SELECT
    ae.work_id,
    ae.is_retracted,
    ae.publication_year,
    ae.author_id,
    ae.author_name,
    CAST(REGEXP_EXTRACT(JSON_VALUE(inst, '$.id'), r'I(\d+)') AS INT64) AS institution_id,
    JSON_VALUE(inst, '$.display_name') AS institution_name,
    JSON_VALUE(inst, '$.country_code') AS country_code,
    ae.primary_source
  FROM authorships_exploded ae,
  UNNEST(ae.institutions) AS inst
),

-- Step 5: Extract journal_id from source
final_pre_filtered AS (
  SELECT
    *,
    CAST(REGEXP_EXTRACT(JSON_VALUE(primary_source, '$.id'), r'S(\d+)') AS INT64) AS journal_id
  FROM institutions_exploded
)

-- Final output: only include journal_ids that match sources_info
SELECT
  work_id,
  is_retracted,
  publication_year,
  author_id,
  author_name,
  institution_id,
  institution_name,
  country_code
FROM final_pre_filtered
WHERE journal_id IN (SELECT source_id FROM sources_info_filtered);


### works_topics

Table Name: `project_name.dataset_name.works_topics` \
Purpose: Contains works topics information. \
Partitioned by: publication_year \
Clustered by: work_id, subfield_id, field_id, domain_id \
Rows: There is one row for each paper, each topic.  \
We are discarding papers with missing publication year, that are not articles or preprints (such as book chapters or repositories), that have been retracted,, that have not been published in journal or conferences, and with missing topics information. Note that each topic belongs to just one subfiled, one field, and one domain. \
Columns:
- `work_id` (INT64): Openalex unique paper ID. Identifies the paper in the Openalex database.
- `is_retracted` (INT64): 1 if the paper has been retracted, 0 otherwise.
- `publication_year` (DATE): The year in which the paper was published.
- `topic_id` (INT64): Unique identifier for the topic that the paper belongs to.
- `topic_score` (FLOAT64): A score between 0 and 1 representing the relevance or strength of the paper to this topic.
- `subfield_id` (INT64): Unique identifier for the subfield in which the paper's topic resides.
- `field_id` (INT64): Unique identifier for the broader research field that includes the subfield.
- `domain_id` (INT64): Unique identifier for the research domain to which the paper belongs.

In [ ]:
CREATE TABLE `project_name.dataset_name.works_topics_retracted`
PARTITION BY publication_year
AS

WITH sources_info_filtered AS (
  SELECT source_id
  FROM mobs-jessica.openalex.sources_info 
  WHERE source_type IN ('journal','conference')
),

works_topics_retracted_test AS (
  SELECT
    CAST(REGEXP_EXTRACT(json_value(work, '$.id'), r'W(\d+)') AS INT64) AS work_id,
    CAST(json_value(work, '$.is_retracted') = 'true' AS INT64) AS is_retracted,
    CAST(CONCAT(CAST(EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', json_value(work, '$.publication_date'))) AS STRING), '-01-01') AS DATE) AS publication_year,
    CAST(REGEXP_EXTRACT(json_value(topic, '$.id'), r'T(\d+)') AS INT64) AS topic_id,
    CAST(json_value(topic, '$.score') AS FLOAT64) AS topic_score,
    CAST(REGEXP_EXTRACT(json_value(topic, '$.subfield.id'), r'(\d+)') AS INT64) AS subfield_id,
    CAST(REGEXP_EXTRACT(json_value(topic, '$.field.id'), r'(\d+)') AS INT64) AS field_id,
    CAST(REGEXP_EXTRACT(json_value(topic, '$.domain.id'), r'(\d+)') AS INT64) AS domain_id,
    CAST(
      IF(
        JSON_EXTRACT(work, '$.primary_location') IS NOT NULL AND 
        JSON_EXTRACT(work, '$.primary_location.source') IS NOT NULL,
        REGEXP_EXTRACT(JSON_EXTRACT_SCALAR(JSON_EXTRACT(work, '$.primary_location.source'), '$.id'), r'S(\d+)'),
        NULL
      ) 
    AS INT64) AS journal_id
  FROM 
    `project_name.dataset_name.works`,
  UNNEST(JSON_EXTRACT_ARRAY(work, '$.topics')) AS topic
  WHERE 
    json_value(work, '$.publication_year') IS NOT NULL  
    AND json_value(work, '$.publication_year') != 'NaN'
    AND json_value(work, '$.type') IN ('article', 'preprint')
    AND LENGTH(CAST(EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', json_value(work, '$.publication_date'))) AS STRING)) = 4
)

SELECT
    work_id,
    is_retracted,
    publication_year,
    topic_id,
    topic_score,
    subfield_id,
    field_id,
    domain_id
FROM works_topics_retracted_test
WHERE journal_id IN (SELECT source_id FROM sources_info_filtered);

## Sources

Creation of 1 tables from sources table:
* sources_info

Table Name: `project_name.dataset_name.sources_info` \
Purpose: Contains sources information. \
Rows: Each row correspond to one source.\ 
Columns:
- `source_id` (INT64): Unique identifier for the source.
- `source_name` (STRING): Name of the source.
- `source_type` (STRING): The type of source, which will be one of: journal, repository, conference, ebook platform, book series, metadata, or other.
- `host_organization` (STRING): Unique identifier of the source host_organization. It can be either a publisher or an institution.
- `host_organization_name` (STRING): Name of the source host_organization. 

In [ ]:
CREATE TABLE `project_name.dataset_name.sources_info`
AS
SELECT
  CAST(REGEXP_EXTRACT(json_value(source, '$.id'), r'S(\d+)') AS INT64) AS source_id,
  CAST(json_value(source, '$.display_name') AS STRING) AS source_name,
  CAST(json_value(source, '$.type') AS STRING) AS source_type,
  CAST(json_value(source, '$.host_organization') AS STRING) AS host_organization,
  CAST(json_value(source, '$.host_organization_name') AS STRING) AS host_organization_name,
FROM 
  `project_name.dataset_name.sources`;